# RY Cnc · a century of glass

This is the "before even that" step of [IDEAS.md idea 2](../IDEAS.md): before forming any
opinion about Boyajian's Star, *pull a DASCH light curve for a star we already understand and
look at what 110 years of plates actually looks like* — including the plates themselves, as
images.

**The target.** RY Cancri, an Algol-type eclipsing binary and the star the
[official DASCH tutorial](https://dasch.cfa.harvard.edu/dr7/rycnc) uses. Two stars orbit each
other every **1.092943 days**; once per orbit the fainter one blocks the brighter one and the
system drops by over a magnitude for a few hours. That period is our **ground truth**: if a
century of hand-carried glass photographs, measured by a modern pipeline, folds cleanly at a
period known to six decimal places, the machinery works (repo rule 5: reproduce a known answer
before chasing an unknown one).

**What we'll see, literally:**

1. the *coverage* — when Harvard photographed this patch of sky, 1886–1990, and the hole in
   the middle of it (the Menzel Gap, the villain of the Boyajian dispute)
2. the raw and cleaned **light curve** — 104 years on one x-axis
3. the phase **fold** at the known period — the ground-truth check
4. actual **plate cutouts** across the decades — scratches, trails and all
5. one night in February 1947 where the eclipse is visible **by eye**, plate vs. plate
6. a **whole glass plate**, 6.5° of sky photographed in 1915

📖 *Resources:* [DASCH DR7](https://dasch.cfa.harvard.edu/dr7/) ·
[daschlab API](https://daschlab.readthedocs.io/) ·
[lightcurve reduction guide](https://dasch.cfa.harvard.edu/dr7/reduce-lightcurve)

## 1. What DASCH is, in one paragraph

From ~1880 to 1990 Harvard College Observatory photographed the sky onto ~550,000 glass
plates, from stations in both hemispheres, with dozens of different telescopes and lenses.
**DASCH** (Digital Access to a Sky Century @ Harvard) scanned them; the final data release,
DR7 (2024), holds 23.6 billion photometric measurements of 252 million sources. A "light
curve" here means: for each plate that covers your star, one brightness measurement — or, if
the star was too faint for that plate, an upper limit. The photometry is *photographic*:
roughly B-band (blue), with honest scatter around **0.1 mag**. This is coarse, century-long,
large-amplitude astronomy — the exact opposite of the ppm-level transit work elsewhere in
this repo.

**Plumbing note.** `daschlab` works through a *session*: a directory where every query result,
light curve, cutout and mosaic is cached as a file. Ours lives under `data/cache/` (gitignored,
`make clean-cache` deletes it). The first run downloads ~30 MB and takes a few minutes; after
that everything below runs from disk.

In [ ]:
import os

# Modest thread caps -- daschlab is network-bound, but keep BLAS from grabbing every core.
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "4")

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from astropy.wcs import WCS

import daschlab

sess = daschlab.open_session("../data/cache/dasch/ry_cnc")
sess.select_target(name="RY Cnc")     # resolved via SIMBAD
sess.select_refcat("apass")           # the recommended reference catalog

target_pos = sess.query().pos_as_skycoord()
print(f"target: RA {target_pos.ra.deg:.5f}  Dec {target_pos.dec.deg:+.5f}")
print(f"reference catalog sources within the ~10' query region: {len(sess.refcat())}")

## 2. Coverage: when was this patch of sky photographed?

An **exposure** is one photograph on one plate (some plates carry several). Asking for every
exposure that covers RY Cnc returns *thirteen thousand* of them — and the histogram of their
dates is itself a piece of history. Plate-taking ramps up through the early 1900s, roars
through the 1930s–40s, then **collapses in 1953**, when director Donald Menzel curtailed the
photographic programme to save money. Coverage only resumes around 1969. That hole is the
**Menzel Gap**, and it is exactly where the [Boyajian's Star dimming
dispute](../IDEAS.md) lives: a calibration step across the gap can masquerade as a secular
trend. We will meet it again.

In [ ]:
exposures = sess.exposures()

# A few plates have no recorded date: their obs_date is *masked*, and .jyear would
# silently render them as J2000. Trap for the unwary -- filter on the mask first.
dated = ~exposures["obs_date"].mask
years = exposures["obs_date"][dated].jyear

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.hist(years, bins=np.arange(1880, 1995, 1), color="#39516b")
ax.axvspan(1953, 1969, color="orange", alpha=0.18)
ax.text(1961, ax.get_ylim()[1] * 0.75, "Menzel\nGap", ha="center", color="darkorange")
ax.set_xlabel("Year")
ax.set_ylabel("Exposures / year")
ax.set_title(f"{len(exposures):,} exposures cover RY Cnc ({years.min():.0f}\u2013{years.max():.0f})")
plt.show()

Those exposures were not taken by one instrument. `series_info()` summarizes the *plate
series* — each is a distinct telescope or camera, from 1-inch patrol lenses to a 24-inch
reflector, each with its own plate scale, depth and quirks. This heterogeneity is the whole
character of the dataset: a DASCH light curve is a committee of dozens of instruments spanning
a century, and cleaning it means deciding which committee members to trust.

In [ ]:
info = exposures.series_info()
info.sort("count", reverse=True)
info[:12]   # the twelve series that photographed this field most often

## 3. The light curve, raw

`sess.lightcurve(0)` fetches the light curve of the catalog source nearest our target — one
row per exposure that measured (or failed to measure) it. Two kinds of rows matter:

- **detections** — the star was measured; `magcal_magdep` holds the calibrated magnitude
  (an astropy `Quantity` in mag, so comparisons need `* u.mag`);
- **nondetections** — the star was below that plate's limit; the magnitude is *masked* and
  `limiting_mag_local` says how deep the plate went.

Plotted raw, warts and all — every point below is a measurement of starlight that left
RY Cnc before anyone alive today was born:

In [ ]:
lc = sess.lightcurve(0)

is_det = ~lc["magcal_magdep"].mask
t_all = lc["time"].jyear
mag_det = np.asarray(lc["magcal_magdep"], dtype=float)

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(t_all[is_det], mag_det[is_det], s=6, color="#39516b", label=f"{is_det.sum()} detections")
ax.scatter(
    t_all[~is_det], np.asarray(lc["limiting_mag_local"], dtype=float)[~is_det],
    s=3, marker="v", color="lightgray", label=f"{(~is_det).sum()} upper limits",
)
ax.invert_yaxis()
ax.set_xlabel("Year")
ax.set_ylabel("Photographic magnitude")
ax.set_title("RY Cnc, raw: every DASCH measurement, 1886\u20131990")
ax.legend(loc="lower right")
plt.show()

## 4. Cleaning: the rejection paradigm

The [reduction guide](https://dasch.cfa.harvard.edu/dr7/reduce-lightcurve) is blunt: raw DASCH
light curves *must* be cleaned, and quotes the Anna Karenina principle — good points are all
alike, bad points are each bad in their own way (plate defects, blends, bad astrometric
solutions, emulsion problems...). Instead of deleting rows, daschlab sets bits in a `reject`
column, so every cut is reversible and taggable. `apply_standard_rejections()` applies the
standard AFLAGS quality cuts (high background, large RMS, too close to the plate limit, ...).
It is deliberately conservative — the guide warns it will not fully clean the data, and we'll
see a few survivors below.

In [ ]:
lc.apply_standard_rejections()

detections = lc.keep_only.nonrej_detected()   # drops rejected rows and nondetections

mags = np.asarray(detections["magcal_magdep"], dtype=float)
t_det = detections["time"].jyear

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(t_det, mags, s=6, color="#39516b")
ax.invert_yaxis()
ax.set_xlabel("Year")
ax.set_ylabel("Photographic magnitude")
ax.set_title(f"RY Cnc, cleaned: {len(detections)} detections over {t_det.max() - t_det.min():.0f} years")
plt.show()

print(f"typical (median) magnitude: {np.median(mags):.2f}")
print(f"faintest surviving detection: {mags.max():.2f} \u2014 these are the eclipses")

The band at ~13.3 is the system's normal brightness (photographic ≈ B band, so fainter than
the V magnitudes catalogs quote). The stragglers *below* the band, one to two magnitudes
faint, are not junk — they are the eclipses, caught by chance whenever a plate happened to be
exposed during the right few hours. The scatter of the band itself, roughly ±0.15 mag, is the
honest per-plate precision of century-old photography.

### Poke at it yourself

`lc.plot()` opens daschlab's native interactive plot (Bokeh): **hover any point** to see its
date, magnitude, plate series and flags; box-zoom into the Menzel Gap; find the eclipse
points. This is the single best way to build a feel for the data.

> Optional, deeper interactivity: in JupyterLab with the WorldWide Telescope extension,
> `await sess.connect_to_wwt()` lets you click sources on a sky map and pull their light
> curves. Not needed for anything below.

In [ ]:
from bokeh.io import output_notebook

output_notebook()   # daschlab's plots are Bokeh; this makes them render inline

lc.summary()
lc.plot()   # interactive: hover for per-point details, drag to box-zoom

## 5. The ground-truth check: fold at the known period

RY Cnc's orbital period is **1.092943 d** ([DASCH tutorial](https://dasch.cfa.harvard.edu/dr7/rycnc)).
If the century of measurements is sound, folding at that period should stack every
chance-caught eclipse from 1886 to 1990 into one sharp dip at phase 0. There is no fitting
here — the period is taken as known, and the epoch is anchored to the faintest point.

In [ ]:
PERIOD_D = 1.092943                      # days, known ground truth
t_jd = np.asarray(detections["time"].jd)
t0 = t_jd[np.argmax(mags)]               # anchor phase 0 on the deepest point

phase = ((t_jd - t0) / PERIOD_D + 0.5) % 1.0 - 0.5

fig, ax = plt.subplots(figsize=(9, 4.5))
sc = ax.scatter(phase, mags, s=8, c=t_det, cmap="viridis")
ax.invert_yaxis()
ax.set_xlabel(f"Phase (P = {PERIOD_D} d)")
ax.set_ylabel("Photographic magnitude")
ax.set_title("104 years of plates, folded at the known binary period")
fig.colorbar(sc, ax=ax, label="Year of the plate")
plt.show()

in_eclipse = mags > 14.0
print(f"detections fainter than mag 14: {in_eclipse.sum()}, "
      f"of which {np.sum(np.abs(phase[in_eclipse]) < 0.15)} land within 0.15 of phase 0")

That is the check passed: the eclipses line up at phase 0 whether the plate was taken in 1918
or 1980 — and the colorbar shows the agreement is not driven by any one era. A period wrong by
even one part in 100,000 would smear eclipses by half a cycle across this baseline; this is
what makes century data powerful for *periods* even at 0.1 mag precision.

Note the handful of impossibly *bright* survivors (one claims magnitude ~9). Those are
exactly what `apply_standard_rejections()` warned it would miss — likely blends or plate
defects — and on a real project they'd be hunted down one by one, cutout in hand. Which
brings us to the part this notebook is really for.

## 6. Seeing the plates

Every number above traces back to a piece of glass in a Cambridge, MA basement. daschlab will
cut a 20′ stamp around the target out of any exposure's scan (~1.4 MB each, cached after the
first download). Below: one plate per era, chosen as the deepest available in each window.
These are *photographs*, with photographic personalities — expect trailed images, scratches,
halation rings around bright stars, and wildly different depths. The red circle is RY Cnc.

In [ ]:
def show_cutout(ax, exp_id, zoom=None):
    """Fetch (or reuse) the cutout for exposure `exp_id` and render it on `ax`."""
    relpath = sess.cutout(exp_id)
    if relpath is None:
        ax.set_axis_off()
        ax.set_title(f"exp {exp_id}: no cutout available", fontsize=9)
        return
    with fits.open(sess.path(relpath)) as hdul:
        data = hdul[0].data.astype(float)
        wcs = WCS(hdul[0].header)
        date = hdul[0].header["DATE-OBS"][:10]
    x, y = wcs.world_to_pixel(target_pos)
    if zoom is not None:
        x0, y0 = int(x), int(y)
        data = data[y0 - zoom : y0 + zoom, x0 - zoom : x0 + zoom]
        x = y = zoom
    lo, hi = ZScaleInterval().get_limits(data)
    ax.imshow(data, cmap="gray_r", vmin=lo, vmax=hi, origin="lower")
    ax.plot(x, y, "o", mfc="none", mec="red", ms=14, mew=1.2)
    row = exposures[exposures["local_id"] == exp_id][0]
    ax.set_title(f"{date} \u00b7 plate {row['series']}{row['platenum']}", fontsize=9)
    ax.set_xticks([]), ax.set_yticks([])


# One representative exposure per ~20-year window: the detection whose plate went deepest.
det_years = np.asarray(detections["time"].jyear)
lim = np.asarray(detections["limiting_mag_local"], dtype=float)
picks = []
for lo_y, hi_y in [(1880, 1900), (1900, 1920), (1920, 1940), (1940, 1955), (1955, 1975), (1975, 1995)]:
    window = (det_years >= lo_y) & (det_years < hi_y)
    if window.any():
        best = np.flatnonzero(window)[np.argmax(lim[window])]
        picks.append(int(detections["exp_local_id"][best]))

fig, axes = plt.subplots(2, 3, figsize=(12, 8.5))
for ax, exp_id in zip(axes.flat, picks):
    show_cutout(ax, exp_id)
for ax in axes.flat[len(picks):]:
    ax.set_axis_off()
fig.suptitle("RY Cnc (red circle), one plate per era \u2014 20\u2032 cutouts", y=0.99)
fig.tight_layout()
plt.show()

## 7. Watching an eclipse happen, by eye

The best part of having images: we can *watch the binary eclipse* between two photographs. The
code below looks for a pair of plates from the same telescope (the 16-inch Metcalf, series
`mc`) taken within a couple of nights of each other — one catching RY Cnc in eclipse, one out
of it. The star visibly fades relative to its neighbours; the neighbours are the control.

In [ ]:
mc = detections[np.asarray(detections["series"]) == "mc"]
mc_mag = np.asarray(mc["magcal_magdep"], dtype=float)
mc_jd = np.asarray(mc["time"].jd)

pair, best_dt = None, np.inf
for i in np.flatnonzero(mc_mag > 14.5):                     # candidate in-eclipse plates
    dt = np.abs(mc_jd - mc_jd[i])
    partners = np.flatnonzero((dt > 0.2) & (dt < 3.0) & (mc_mag < 13.5))
    if partners.size and dt[partners].min() < best_dt:
        j = partners[np.argmin(dt[partners])]
        pair, best_dt = (int(mc["exp_local_id"][j]), int(mc["exp_local_id"][i])), dt[j]

assert pair is not None, "no close mc-series pair found -- widen the windows above"
out_id, in_id = pair

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, exp_id in zip(axes, [out_id, in_id]):
    show_cutout(ax, exp_id, zoom=110)                        # ~5' stamps
    row = lc[lc["exp_local_id"] == exp_id]
    state = "OUT of eclipse" if exp_id == out_id else "IN eclipse"
    ax.set_xlabel(f"{state}: measured mag {float(row['magcal_magdep'][0].value):.2f}")
fig.suptitle(f"Same telescope, {best_dt:.1f} days apart \u2014 the eclipse, on glass")
fig.tight_layout()
plt.show()

## 8. And finally: a whole plate

Cutouts hide the most striking thing about this data — the sheer *size* of a plate.
`sess.mosaic()` fetches a full-plate scan (here binned 16×16, ~3 MB; full resolution would be
~600 MB). This one, `mc07628`, is an hour-long exposure from the night of 1915 January 11.
The rich knot of stars around the red circle is no plotting artifact — it is the **Beehive
Cluster** (M44, Praesepe); RY Cnc happens to sit on its outskirts.
The value-added FITS even carries extensions listing the exposures and links to photographs
of the plate's paper jacket, where a century of observers' and computers' annotations live.

In [ ]:
mosaic_path = sess.path(sess.mosaic("mc07628", binning=16))

with fits.open(mosaic_path) as hdul:
    plate = np.asarray(hdul[0].data, dtype=float)
    wcs = WCS(hdul[0].header)
    date = hdul[0].header["DATE-OBS"][:10]

ny, nx = plate.shape
corner = wcs.pixel_to_world([0, nx - 1, 0], [0, 0, ny - 1])
width, height = corner[0].separation(corner[1]).deg, corner[0].separation(corner[2]).deg

x, y = wcs.world_to_pixel(target_pos)
lo, hi = ZScaleInterval().get_limits(plate)

fig, ax = plt.subplots(figsize=(12, 9.5))
ax.imshow(plate, cmap="gray_r", vmin=lo, vmax=hi, origin="lower")
ax.plot(x, y, "o", mfc="none", mec="red", ms=16, mew=1.4)
ax.set_xticks([]), ax.set_yticks([])
ax.set_title(f"Plate mc07628, {date}: {width:.1f}\u00b0 \u00d7 {height:.1f}\u00b0 of sky on one piece of glass")
plt.show()

## 9. What 110 years of plates actually looks like

- **The machinery works.** A century of heterogeneous photographs, cleaned with the standard
  rejections, folds cleanly at an externally-known period. Ground truth reproduced.
- **Coverage has *structure*.** The Menzel Gap is not a footnote — it is a wall across the
  1950s–60s visible in a one-line histogram. Any claim about slow trends in DASCH data has to
  survive that discontinuity, which is precisely the Boyajian's Star argument.
- **~0.1–0.15 mag scatter is real**, and the standard cleaning is deliberately incomplete: a
  mag-9 ghost survived into our cleaned curve. Per-point forensics (with cutouts) is part of
  any serious DASCH result.
- **The images are the escape hatch.** When a data point looks strange, you can go look at
  the actual photograph — trailed stars, scratches and all. Almost no other archive lets an
  outlier be adjudicated by eye against the original observation.

**Next step** (per [IDEAS.md](../IDEAS.md)): point this exact machinery at KIC 8462852,
reproduce Schaefer's dimming measurement *and* the Hippke/Lund counter-analysis, and form a
defended opinion on who is right. The Menzel Gap histogram above is the first exhibit either
way.

*Reproducibility: everything downloads into `data/cache/dasch/ry_cnc/` (~30 MB); re-runs are
offline and take seconds. Delete with `make clean-cache`.*